In [3]:
%load_ext juliacall

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [8]:

# Latest attempt to make a Matrix object { Julia is integrated into Python using juliacall } 

# Functions like ref,rref,inverse,det and solveLSE are present in the phase-3 of the project, and they will be bridged behind the scenes using juliacall to ensure speed and efficiency.


from juliacall import Main as jl

jl.seval("""

    function matmult( x:: Matrix{<: Number} , y:: Matrix{<: Number})   
        x_dim =  size(x)
        y_dim =  size(y)
        if x_dim[2] != y_dim[1] 
            throw(ArgumentError("Invalid input detectet/dimensions not matching"))
        else 
            # xy = Matrix{Number}(undef,x_dim[1],y_dim[2])
             xy = zeros(Number,x_dim[1],y_dim[2])
            @inbounds @simd for i in 1:1:y_dim[2] 
           @inbounds @simd     for j in 1:1:x_dim[2] 
               @inbounds @simd   for k in 1:1:x_dim[1] 
                        xy[k,i] += x[k,j]*y[j,i]
                    
                                    end
                                end
                            end
            return xy
        end
    end
   """    
        )

jl.seval("""

function add(x:: Matrix{<:Number},y:: Matrix{<: Number})
    x_dim = size(x)
    y_dim = size(y)
    if x_dim != y_dim
        throw(ArgumentError("Dimension Mismatch"))
    else 
        xy = zeros(Number,x_dim[1],x_dim[2])
      @inbounds @simd  for i = 1:1:x_dim[2] # i is  column pointer
         @inbounds @simd   for j = 1:1:x_dim[1] # j is row pointer
                xy[j,i] = x[j,i]+y[j,i]
            end
        end       
    end
    return xy
end




""")

jl.seval("""

function sub(x:: Matrix{<:Number},y:: Matrix{<: Number})
    x_dim = size(x)
    y_dim = size(y)
    if x_dim != y_dim
        throw(ArgumentError("Dimension Mismatch"))
    else 
        xy = zeros(Number,x_dim[1],x_dim[2])
      @inbounds @simd  for i = 1:1:x_dim[2] # i is  column pointer
         @inbounds @simd   for j = 1:1:x_dim[1] # j is row pointer
                xy[j,i] = x[j,i] - y[j,i]
            end
        end       
    end
    return xy
end




""")

jl.seval("""
function transpose_(x::Matrix{<:Number})
    dims = size(x)
    xT =zeros(Number, dims[2],dims[1])
    @inbounds @simd for i = 1:1:dims[1] 
       @inbounds @simd for j in 1:1:dims[2]  
                              xT[j,i] = x[i,j]
        end
    end
return xT
end

function trace(M_raw ::Matrix{<:Number})
    M= M_raw
    sz = size(M)

    if sz[2] != sz[1]
        return throw(ArgumentError("Non-Square Matrices dont have Trace"))
    else
        trace = 0
        for i in 1:sz[1]
            trace += M[i, i]
        end
    end
    return trace
end

"""
        )


class mat ():
    def __init__(self,*rv): # rv = row vectors
        self.__rv = jl.convert(jl.Matrix[jl.Number],jl.transpose(jl.hcat(*rv)))
        
    def iter(self):
        return self.__rv  
        
    def __repr__(self):
        # Leveraging Julia's own function 
        return jl.sprint(jl.show, jl.MIME("text/plain"), self.__rv)
        
    def dim(self) :
        return jl.size(self.iter())
        
    def transpose(self):
       return jl.transpose_(self.iter())
    
    def __mul__(u,v): 
        return jl.matmult(u.iter(),v.iter())

    ''' in julia matrix multiplication
    first column is build first ( using the 3rd loop { k is changing fastest})
    then using 2nd loo the column is updated again and again ( cuz j changes)
    then i changes and we move to the next column ( then above 2 steps are repeated)'''
    def __add__(u,v):
        return jl.add(u.iter(),v.iter())
    def __sub__(u,v):
        return jl.sub(u.iter(),v.iter())
    def trace(u):
        return jl.trace(u.iter())
    def ref(u):
        return jl.ref(u.iter())
    def rref(u):
        return jl.rref(u.iter())
    def solveLSE(u):
        return jl.solveLSE(u.iter())
    def inverse(u):
        return jl.inverse(u.iter())
    def det(u):
        return jl.det(u.iter())
        
            
     # ==========================================
# TEST CASES ARE GENERATED BY LLM, AND THEY ARE NOT EXHAUSTIVE. THEY ARE MEANT TO SERVE AS A BASIC SANITY CHECK FOR THE CORE BACKEND ALGEBRA.
# ==========================================

# 1. Initialize two 2x2 matrices
# (Passing row vectors into the mat constructor)
A = mat([1.0, 2.0], [3.0, 4.0])
B = mat([5.0, 6.0], [7.0, 8.0])

print("--- Test Case 1: Structural Properties & Representation ---")
print("Matrix A Printout:")
print(A)
print(f"\nDimensions of Matrix A: {A.dim()}")
print(f"Trace of Matrix A: {A.trace()}")
print(f"Transpose of Matrix A:\n{A.transpose()}\n")


print("--- Test Case 2: Element-Wise Matrix Arithmetic ---")
print(f"Addition (A + B):\n{A + B}\n")
print(f"Subtraction (A - B):\n{A - B}\n")


print("--- Test Case 3: High-Performance Matrix Multiplication ---")
# Compares native multiplication across your fast loops
try:
    product = A * B
    print(f"Multiplication Matrix Product (A * B):\n{product}\n")
    print("✅ All core backend algebra tests passed successfully!")
except Exception as e:
    print(f"❌ Verification failed due to error: {e}")   
    
        

--- Test Case 1: Structural Properties & Representation ---
Matrix A Printout:
2×2 Matrix{Number}:
 1.0  2.0
 3.0  4.0

Dimensions of Matrix A: (2, 2)
Trace of Matrix A: 5.0
Transpose of Matrix A:
Number[1.0 3.0; 2.0 4.0]

--- Test Case 2: Element-Wise Matrix Arithmetic ---
Addition (A + B):
Number[6.0 8.0; 10.0 12.0]

Subtraction (A - B):
Number[-4.0 -4.0; -4.0 -4.0]

--- Test Case 3: High-Performance Matrix Multiplication ---
Multiplication Matrix Product (A * B):
Number[19.0 22.0; 43.0 50.0]

✅ All core backend algebra tests passed successfully!
